# ⛏️ Baritone Backend — Minecraft Java AI Pathfinding on Colab

Runs a headless Minecraft Java client with [Baritone](https://github.com/brg123/Baritone) mod on Colab.
Exposes an HTTP API via ngrok for the Discord bot to send commands.

## What Baritone does
- **Pathfinding IA** (A*) — navigates automatically, avoids lava, water, falls
- `#goto x z` — go to coordinates
- `#mine diamond_ore` — auto-mine specific ores
- `#follow player` — follow a player
- `#build` — auto-build structures
- `#explore` — explore the world automatically

## Architecture
```
Discord Bot → POST /command → Colab FastAPI → xdotool types in MC chat
                                     ↓
                              Minecraft Java + Baritone
                                     ↓
                              Log file → POST /status → Bot reads status
```

**Instructions:**
1. Runtime > Change runtime type > CPU (no GPU needed for MC)
2. Execute all cells in order
3. Set your Minecraft server IP and credentials in Cell 2

In [ ]:
# Cell 1: Install Java 21 + tools
!apt-get update -qq && apt-get install -y -qq openjdk-21-jre-headless xvfb xdotool screen
!pip install fastapi uvicorn pyngrok requests

import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']
!java -version

In [ ]:
# Cell 2: Configuration
MC_VERSION = '1.21.1'  # Minecraft version
MC_SERVER = ''  # <-- Your server IP:port (e.g. 'play.example.com:25565')
MC_USERNAME = ''  # <-- Microsoft username (or email)
MC_PASSWORD = ''  # <-- Microsoft password (leave empty for offline/cracked)
OFFLINE_MODE = True  # True = cracked/offline, False = premium

NGROK_AUTHTOKEN = ''  # <-- Paste your ngrok authtoken
BOT_WEBHOOK_URL = ''  # <-- Bot webhook URL for auto-URL update
API_PORT = 9000

# Baritone settings
BARITONE_SETTINGS = {
    'acceptableThrowawayItems': 'cobblestone,dirt,gravel,andesite,granite,diorite',
    'freeLook': 'true',
    'mapAutoUpdate': 'true',
    'mineScanRadius': '64',
    'pathThroughWater': 'true',
    'allowParkour': 'true',
    'allowBreak': 'true',
    'allowPlace': 'true',
    'allowInventory': 'true',
    'allowSprint': 'true',
}

print(f'Config: MC {MC_VERSION}, Server={MC_SERVER}, Offline={OFFLINE_MODE}')

In [ ]:
# Cell 3: Download Minecraft + Fabric + Baritone
import os, subprocess, time

MC_DIR = '/root/minecraft'
os.makedirs(MC_DIR, exist_ok=True)
os.makedirs(f'{MC_DIR}/mods', exist_ok=True)
os.makedirs(f'{MC_DIR}/config', exist_ok=True)

# Download Fabric loader
print('⏳ Downloading Fabric installer...')
os.system(f'cd {MC_DIR} && wget -q -O fabric-installer.jar https://maven.fabricmc.net/net/fabricmc/fabric-installer/1.0.1/fabric-installer-1.0.1.jar')

# Run Fabric installer
print(f'⏳ Installing Fabric for MC {MC_VERSION}...')
os.system(f'cd {MC_DIR} && java -jar fabric-installer.jar client -dir {MC_DIR} -mcversion {MC_VERSION} -loader 0.16.9')
print('✅ Fabric installed')

# Download Baritone
print('⏳ Downloading Baritone...')
# Baritone for Fabric
os.system(f'cd {MC_DIR}/mods && wget -q -O baritone.jar https://github.com/brg123/Baritone/releases/download/v1.21.1/baritone-fabric-1.21.1.jar')
if os.path.exists(f'{MC_DIR}/mods/baritone.jar'):
    print('✅ Baritone downloaded')
else:
    print('⚠️ Baritone download failed — trying alternate URL...')
    os.system(f'cd {MC_DIR}/mods && wget -q -O baritone.jar https://github.com/brg123/Baritone/releases/latest/download/baritone-fabric-{MC_VERSION}.jar')

# Write Baritone config
config_lines = [f'{k}={v}' for k, v in BARITONE_SETTINGS.items()]
with open(f'{MC_DIR}/config/baritone.txt', 'w') as f:
    f.write('\n'.join(config_lines))
print('✅ Baritone config written')

# List mods
print(f'\nMods in {MC_DIR}/mods:')
os.system(f'ls -la {MC_DIR}/mods/')

In [ ]:
# Cell 4: Start Xvfb (virtual display) + Minecraft
import subprocess, time, os, signal

# Start virtual display
os.system('pkill Xvfb 2>/dev/null; sleep 1')
xvfb = subprocess.Popen(['Xvfb', ':99', '-screen', '0', '1280x720x24'],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
os.environ['DISPLAY'] = ':99'
time.sleep(2)
print('✅ Xvfb display started')

# Build launch command
launch_cmd = [
    'java',
    f'-Xmx4G',
    f'-Djava.library.path={MC_DIR}/natives',
    '-cp', f'{MC_DIR}/libraries/*:{MC_DIR}/minecraft.jar',
    'net.fabricmc.loader.impl.launch.knot.KnotClient',
    '--username', MC_USERNAME if not OFFLINE_MODE else f'Bot_{MC_USERNAME}',
    '--version', MC_VERSION,
    '--gameDir', MC_DIR,
    '--assetsDir', f'{MC_DIR}/assets',
    '--assetIndex', MC_VERSION.replace('.', '_'),
    '--uuid', '00000000-0000-0000-0000-000000000000',
    '--accessToken', '0',
    '--server', MC_SERVER,
]

if OFFLINE_MODE:
    launch_cmd.extend(['--offline'])

print(f'⏳ Starting Minecraft {MC_VERSION}...')
print(f'   Server: {MC_SERVER}')
print(f'   Username: Bot_{MC_USERNAME}' if OFFLINE_MODE else f'   Username: {MC_USERNAME}')

# Start Minecraft in screen session
os.system('screen -X -S minecraft quit 2>/dev/null; sleep 1')
os.system(f'screen -dmS minecraft bash -c "DISPLAY=:99 {\" \".join(launch_cmd)} 2>&1 | tee {MC_DIR}/mc_log.txt"')
time.sleep(10)

# Check if running
result = os.system('screen -ls | grep minecraft')
if result == 0:
    print('✅ Minecraft started in screen session')
else:
    print('❌ Minecraft failed to start — check log:')
    os.system(f'tail -20 {MC_DIR}/mc_log.txt')

In [ ]:
# Cell 5: FastAPI control server
from fastapi import FastAPI
from pydantic import BaseModel
import subprocess, os, time, re, threading

app = FastAPI(title='Baritone Control API', version='1.0.0')

class CommandRequest(BaseModel):
    command: str  # e.g. 'goto 100 200', 'mine diamond_ore', 'stop', 'follow Player1'

class SettingsRequest(BaseModel):
    key: str
    value: str

MC_LOG = f'{MC_DIR}/mc_log.txt'

def send_to_mc(text: str):
    """Send text to Minecraft chat via xdotool."""
    # Open chat with 't' key, type command, press Enter
    os.system('xdotool key t')
    time.sleep(0.3)
    # Type the text (escape special chars)
    os.system(f"xdotool type --delay 10 '{text}'")
    time.sleep(0.2)
    os.system('xdotool key Return')

def read_log_tail(lines: int = 50) -> str:
    """Read last N lines from Minecraft log."""
    try:
        with open(MC_LOG, 'r') as f:
            all_lines = f.readlines()
            return ''.join(all_lines[-lines:])
    except:
        return ''

def parse_status(log: str) -> dict:
    """Parse Minecraft log for status info."""
    status = {
        'connected': False,
        'position': None,
        'health': None,
        'food': None,
        'current_task': None,
        'baritone_active': False,
    }
    # Check connection
    if 'Joined the game' in log or 'Connected to' in log:
        status['connected'] = True
    # Parse position from log
    pos_match = re.findall(r'\[Chat\] .*?XYZ: ([\d.\-]+) / ([\d.\-]+) / ([\d.\-]+)', log)
    if pos_match:
        last = pos_match[-1]
        status['position'] = {'x': float(last[0]), 'y': float(last[1]), 'z': float(last[2])}
    # Parse health
    health_match = re.findall(r'Health: ([\d.]+)', log)
    if health_match:
        status['health'] = float(health_match[-1])
    food_match = re.findall(r'Food: ([\d.]+)', log)
    if food_match:
        status['food'] = float(food_match[-1])
    # Baritone status
    if 'Baritone' in log and ('pathing' in log.lower() or 'mining' in log.lower()):
        status['baritone_active'] = True
    task_match = re.findall(r'\[Baritone\] (?:INFO )?(.+)', log)
    if task_match:
        status['current_task'] = task_match[-1]
    return status

@app.get('/health')
async def health():
    return {'status': 'ok', 'mc_version': MC_VERSION, 'server': MC_SERVER}

@app.get('/status')
async def get_status():
    """Get Minecraft + Baritone status from log."""
    log = read_log_tail(100)
    return parse_status(log)

@app.post('/command')
async def send_command(req: CommandRequest):
    """Send a Baritone command to Minecraft.
    Commands are prefixed with # for Baritone.
    Examples: 'goto 100 200', 'mine diamond_ore', 'follow Player1', 'stop', 'explore'
    """
    cmd = req.command.strip()
    if not cmd:
        return {'error': 'Empty command'}, 400
    # Auto-prefix with # if not present (Baritone commands start with #)
    if not cmd.startswith('#'):
        cmd = '#' + cmd
    try:
        send_to_mc(cmd)
        return {'success': True, 'command': cmd, 'sent_at': time.time()}
    except Exception as e:
        return {'error': str(e)}, 500

@app.post('/chat')
async def send_chat(req: CommandRequest):
    """Send a regular chat message (not a Baritone command)."""
    try:
        send_to_mc(req.command)
        return {'success': True, 'message': req.command}
    except Exception as e:
        return {'error': str(e)}, 500

@app.post('/settings')
async def update_setting(req: SettingsRequest):
    """Update a Baritone setting at runtime."""
    cmd = f'#set {req.key} {req.value}'
    try:
        send_to_mc(cmd)
        return {'success': True, 'setting': req.key, 'value': req.value}
    except Exception as e:
        return {'error': str(e)}, 500

@app.get('/log')
async def get_log(lines: int = 50):
    """Get last N lines of Minecraft log."""
    return {'log': read_log_tail(lines)}

@app.post('/stop')
async def stop_baritone():
    """Stop Baritone pathfinding."""
    send_to_mc('#stop')
    return {'success': True, 'message': 'Baritone stopped'}

@app.post('/force-disconnect')
async def force_disconnect():
    """Force disconnect from Minecraft server."""
    os.system('screen -X -S minecraft quit')
    return {'success': True, 'message': 'Minecraft client stopped'}

print('✅ Baritone Control API ready:')
print('  GET  /health        — API health')
print('  GET  /status        — MC + Baritone status')
print('  POST /command       — Send Baritone command (#goto, #mine, #follow)')
print('  POST /chat          — Send chat message')
print('  POST /settings      — Update Baritone setting')
print('  GET  /log?lines=N   — Read MC log')
print('  POST /stop          — Stop Baritone')
print('  POST /force-disconnect — Kill MC client')

In [ ]:
# Cell 6: Start ngrok + API server
from pyngrok import ngrok, conf
import nest_asyncio, threading, uvicorn, requests

if NGROK_AUTHTOKEN:
    conf.get_default().auth_token = NGROK_AUTHTOKEN

ngrok.kill()
import time; time.sleep(2)

tunnel = ngrok.connect(API_PORT, 'http')
BARITONE_URL = tunnel.public_url
print(f'🌐 Baritone API URL: {BARITONE_URL}')
print(f'   → Set BARITONE_URL={BARITONE_URL} in bot .env')

# Notify bot webhook
if BOT_WEBHOOK_URL:
    try:
        resp = requests.post(BOT_WEBHOOK_URL, json={'url': BARITONE_URL, 'type': 'baritone'}, timeout=10)
        print(f'📡 Bot notified: {resp.status_code}')
    except Exception as e:
        print(f'⚠️ Webhook failed: {e}')

# Start server
nest_asyncio.apply()
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=API_PORT, log_level='info')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print(f'✅ API server running on port {API_PORT}')
print(f'   Test: {BARITONE_URL}/health')

In [ ]:
# Cell 7: Keep-alive + auto-regeneration (same pattern as other notebooks)
import time, requests
from datetime import datetime, timedelta

REGENERATE_HOURS = 24
start_time = datetime.now()
regenerate_at = start_time + timedelta(hours=REGENERATE_HOURS)

print(f'🕐 Baritone backend started at {start_time.strftime("%H:%M:%S")}')
print(f'🔄 Will regenerate at {regenerate_at.strftime("%H:%M:%S")}')
print(f'📊 Keep-alive loop (checks every 60s)...')

ok_count = 0
fail_count = 0

try:
    while True:
        now = datetime.now()
        
        # Health check
        try:
            resp = requests.get(f'{BARITONE_URL}/health', timeout=10)
            if resp.status_code == 200:
                ok_count += 1
                status = '✅'
            else:
                fail_count += 1
                status = f'⚠️ {resp.status_code}'
        except:
            fail_count += 1
            status = '❌'
        
        # Status every 5 min
        elapsed = now - start_time
        if int(elapsed.total_seconds()) % 300 == 0 and int(elapsed.total_seconds()) > 0:
            # Also get MC status
            try:
                mc_status = requests.get(f'{BARITONE_URL}/status', timeout=10).json()
                mc_info = f"MC={'online' if mc_status.get('connected') else 'offline'}"
                if mc_status.get('position'):
                    p = mc_status['position']
                    mc_info += f" pos=({p['x']:.0f},{p['y']:.0f},{p['z']:.0f})"
                if mc_status.get('baritone_active'):
                    mc_info += ' baritone=active'
            except:
                mc_info = 'MC=status?'
            print(f'[{now.strftime("%H:%M:%S")}] uptime={int(elapsed.total_seconds()/60)}min ok={ok_count} fail={fail_count} {status} {mc_info}')
        
        # Regenerate ngrok every 24h
        if now >= regenerate_at:
            print('🔄 Regenerating ngrok tunnel...')
            ngrok.kill()
            time.sleep(3)
            new_tunnel = ngrok.connect(API_PORT, 'http')
            BARITONE_URL = new_tunnel.public_url
            print(f'   New URL: {BARITONE_URL}')
            if BOT_WEBHOOK_URL:
                try:
                    requests.post(BOT_WEBHOOK_URL, json={'url': BARITONE_URL, 'type': 'baritone'})
                    print('   📡 Bot notified')
                except Exception as e:
                    print(f'   ⚠️ Webhook failed: {e}')
            regenerate_at = now + timedelta(hours=REGENERATE_HOURS)
        
        time.sleep(60)
except KeyboardInterrupt:
    print('\n⏹️ Stopped')
except Exception as e:
    print(f'\n❌ Error: {e}')
finally:
    print(f'Stats: ok={ok_count} fail={fail_count}')

In [ ]:
# Cell 8 (optional): Test commands
import requests

# Test health
print('Health:', requests.get(f'{BARITONE_URL}/health').json())

# Test status
print('Status:', requests.get(f'{BARITONE_URL}/status').json())

# Send a test command (uncomment to test)
# print('Goto:', requests.post(f'{BARITONE_URL}/command', json={'command': 'goto 0 0'}).json())
# print('Mine:', requests.post(f'{BARITONE_URL}/command', json={'command': 'mine diamond_ore'}).json())
# print('Stop:', requests.post(f'{BARITONE_URL}/stop').json())